In [13]:
import numpy as np
import pandas as pd
import ROOT
import glob
import sys, os, re
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta
from astropy.coordinates import CartesianRepresentation, SkyCoord, angular_separation, EarthLocation, GCRS, ITRS
from astropy.coordinates import Longitude, Latitude, AltAz
from astropy import units as u
from shapely.geometry import Point, Polygon
from shapely import vectorized
import psutil

In [14]:

# Load the table once globally
PATH_EFFAREA = "/home/users/l/lebrun/results/effArea/effectiveArea_weighted.npz"
PATH_BINS = "/home/users/l/lebrun/results/effArea_bins_info_HET.npz"
# Load the effective area table
EFF_AREA_NPZ = np.load(PATH_EFFAREA)
EFF_AREA_TABLE = EFF_AREA_NPZ['effArea'] # Shape (n_E_bins, n_costheta_bins, n_phi_bins) ~ (20, 10, 360)

# Get bins edges and centers
ENERGY_BINS_EDGES = EFF_AREA_NPZ['E_bins_edge']
ENERGY_BINS_CENTERS = np.sqrt(ENERGY_BINS_EDGES[:-1] * ENERGY_BINS_EDGES[1:])  # GeV
COSTHETA_BINS_EDGES = EFF_AREA_NPZ['theta_bins_edge']
PHI_BINS_EDGES = EFF_AREA_NPZ['phi_bins_edge']

# SAA Region

saa_lat_2016 = [
    -8.136145, -5.522644, -2.909143, -0.295642, 2.317859, 4.931360,
    -1.170294, -5.924736, -10.080192, -13.636662, -16.594145, -18.952644,
    -32.560543, -34.555897, -36.591114, -38.661793, -40.743484, -42.791676,
    -44.741795, -46.509228, -47.989288, -49.057251, -49.568325, -49.357666,
    -48.240376, -46.011497, -42.446022, -37.298885, -30.304964, -21.179083
]

saa_lon_2016 = [
    -95.0, -84.0, -73.0, -62.0, -51.0, -40.0,
    -35.0, -29.0, -23.0, -17.0, -11.0, -5.0,
    -5.0, -10.0, -15.0, -20.0, -25.0, -30.0,
    -35.0, -40.0, -45.0, -50.0, -55.0, -60.0,
    -65.0, -70.0, -75.0, -80.0, -85.0, -90.0
]

saa_polygon_2016 = list(zip(saa_lon_2016, saa_lat_2016))
saa_polygon_shape = Polygon(saa_polygon_2016)  # Define SAA region

In [19]:
def met_to_utc(seconds, microseconds):
    seconds = np.asarray(seconds)
    microseconds = np.asarray(microseconds)
    delta_sec = TimeDelta(seconds, format='sec')
    delta_us = TimeDelta(microseconds * 1e-6, format='sec')
    epoch = Time("2013-01-01T00:00:00", scale="utc")
    return epoch + delta_sec + delta_us

def mask_saa(df):
    """
    Mask out timestamps where the satellite is within the SAA region.
    
    Parameters:
    - df: pandas.DataFrame with columns ['second', 'microsecond', 'x', 'y', 'z', 'utc']
    
    Returns:
    - df_masked: DataFrame excluding SAA timestamps
    - mask: boolean mask array (True = keep, False = SAA)
    """
    # Convert MET to UTC Time objects
    obstime = df['utc']

    # Cartesian coordinates in meters
    positions = CartesianRepresentation(
        x=df['x'].to_numpy() * u.m,
        y=df['y'].to_numpy() * u.m,
        z=df['z'].to_numpy() * u.m
    )

    # Build GCRS coordinate frame at each time
    gcrs_coords = GCRS(positions, obstime=obstime)

    # Convert to Earth-fixed frame (ITRS), then to geographic coordinates
    itrs_coords = gcrs_coords.transform_to(ITRS(obstime=obstime))
    geodetic = itrs_coords.earth_location.to_geodetic()

    # Extract lat/lon in degrees
    lat = geodetic.lat.to(u.deg).value
    lon = geodetic.lon.to(u.deg).value

    # Define SAA region
    mask_inside_saa = vectorized.contains(saa_polygon_shape, lon, lat)

    # Invert to create a mask of valid points
    valid_mask = ~mask_inside_saa
    df_masked = df[valid_mask].reset_index(drop=True)
    return df_masked, valid_mask

def load_filtered_day_dataframe(day_path, next_day_path, day_utc_start, day_utc_end):
    files = sorted(glob.glob(day_path) + glob.glob(next_day_path))
    rdf = ROOT.RDataFrame("HousekeepingData/SatStatus", files)
    data = rdf.AsNumpy(['DmpHKDSatStatus.fSecond', 'DmpHKDSatStatus.fMicrosecond',
                        'DmpHKDSatStatus.fPosition.fX', 'DmpHKDSatStatus.fPosition.fY', 'DmpHKDSatStatus.fPosition.fZ',
                        "DmpHKDSatStatus.fVelocity.fX", 
                        "DmpHKDSatStatus.fVelocity.fY", 
                        "DmpHKDSatStatus.fVelocity.fZ"])
    df = pd.DataFrame(data)
    df.columns = ['second', 'microsecond', 'x', 'y', 'z', 'vx', 'vy', 'vz']
    df = df.sort_values(by=['second', 'microsecond']).drop_duplicates('second', keep='first')

    # UTC filtering to ensure no overlap between days
    utc_time = met_to_utc(df['second'], df['microsecond'])
    mask = (utc_time >= day_utc_start) & (utc_time < day_utc_end)
    df = df.loc[mask].reset_index(drop=True)
    df['utc'] = utc_time[mask]

    delta_t = (df['second'].to_numpy()[1:] - df['second'].to_numpy()[:-1]) \
            + 1e-6 * (df['microsecond'].to_numpy()[1:] - df['microsecond'].to_numpy()[:-1])
    delta_t = np.append(delta_t, 0.0)
    df['Delta_t'] = delta_t

    return df


def parse_par_file(filename):
    """
    Parse a .par file and extract RAJ and DECJ values.
    
    Parameters:
    filename (str): Path to the .par file
    
    Returns:
    tuple: (raj_value, decj_value) or (None, None) if not found
    """
    raj_value = None
    decj_value = None
    
    try:
        with open(filename, 'r') as f:
            for line in f:
                line = line.strip()
                
                # Skip empty lines and comments
                if not line or line.startswith('#'):
                    continue
                
                # Split line into components
                parts = line.split()
                if len(parts) < 2:
                    continue
                
                parameter = parts[0].upper()
                
                # Extract RAJ value
                if parameter == 'RAJ':
                    # RAJ value is typically in format HH:MM:SS.ssss
                    raj_value = parts[1]
                
                # Extract DECJ value  
                elif parameter == 'DECJ':
                    # DECJ value is typically in format +/-DD:MM:SS.ssss
                    decj_value = parts[1]
    
    except FileNotFoundError:
        print(f"Error: Could not find file {filename}")
        return None, None
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        return None, None
    
    return raj_value, decj_value

def convert_ra_to_decimal(ra_str):
    """
    Convert RA string from HH:MM:SS.ssss format to decimal hours.
    
    Parameters:
    ra_str (str): RA in HH:MM:SS.ssss format
    
    Returns:
    float: RA in decimal hours
    """
    if ra_str is None:
        return None
    
    try:
        parts = ra_str.split(':')
        hours = float(parts[0])
        minutes = float(parts[1]) if len(parts) > 1 else 0
        seconds = float(parts[2]) if len(parts) > 2 else 0
        
        decimal_hours = hours + minutes/60.0 + seconds/3600.0
        return decimal_hours
    except (ValueError, IndexError):
        print(f"Error parsing RA: {ra_str}")
        return None

def convert_dec_to_decimal(dec_str):
    """
    Convert DEC string from +/-DD:MM:SS.ssss format to decimal degrees.
    
    Parameters:
    dec_str (str): DEC in +/-DD:MM:SS.ssss format
    
    Returns:
    float: DEC in decimal degrees
    """
    if dec_str is None:
        return None
    
    try:
        # Handle sign
        sign = 1
        if dec_str.startswith('+'):
            dec_str = dec_str[1:]
        elif dec_str.startswith('-'):
            sign = -1
            dec_str = dec_str[1:]
        
        parts = dec_str.split(':')
        degrees = float(parts[0])
        minutes = float(parts[1]) if len(parts) > 1 else 0
        seconds = float(parts[2]) if len(parts) > 2 else 0
        
        decimal_degrees = degrees + minutes/60.0 + seconds/3600.0
        return sign * decimal_degrees
    except (ValueError, IndexError):
        print(f"Error parsing DEC: {dec_str}")
        return None


def effective_area(cos_theta, phi, energy):
    cos_theta = np.asarray(cos_theta)
    energy_idx = np.digitize(energy, ENERGY_BINS_EDGES) - 1
    costheta_idx = np.digitize(cos_theta, COSTHETA_BINS_EDGES) - 1
    phi_idx = np.digitize(phi, PHI_BINS_EDGES) - 1

    valid = (
        (energy_idx >= 0) & (energy_idx < EFF_AREA_TABLE.shape[0]) &
        (costheta_idx >= 0) & (costheta_idx < EFF_AREA_TABLE.shape[1]) &
        (phi_idx >= 0) & (phi_idx < EFF_AREA_TABLE.shape[2])
    )

    eff_area = np.zeros_like(cos_theta, dtype=float)

    # Only fill valid entries
    if valid.any():
        # Since energy_idx is scalar, just broadcast it:
        e_idx = energy_idx
        c_idx = costheta_idx[valid]
        p_idx = phi_idx[valid]

        eff_area[valid] = EFF_AREA_TABLE[e_idx, c_idx, p_idx]

    return eff_area

def transform_vectors_to_galactic(vectors):
    # Input: shape (N, 3)
    coords_icrs = SkyCoord(
        x=vectors[:, 0], y=vectors[:, 1], z=vectors[:, 2],
        representation_type=CartesianRepresentation,
        frame='icrs'
    )
    coords_gal = coords_icrs.transform_to('galactic')
    return coords_gal.cartesian.xyz.value.T  # shape (N, 3)

def compute_exposure_map_roi(df, energy,
                             roi_center_lon, roi_center_lat, roi_side_deg=4, fov_angle_deg=60):

    """ Compute the exposure map for a given region of interest (ROI) in galactic coordinates.
    Args:
        df (pd.DataFrame): DataFrame containing satellite state data with columns ['x', 'y', 'z', 'vx', 'vy', 'vz', 'utc'].
        energy (float): Energy in GeV for which to compute the exposure.
        roi_center_lon (float): Center longitude of the ROI in degrees.
        roi_center_lat (float): Center latitude of the ROI in degrees.
        roi_side_deg (float): Side length of the square ROI in degrees.
        fov_angle_deg (float): Field of view angle in degrees.
    Returns:
        dict: A dictionary containing the exposure map and grid coordinates.
    """

    # Full resolution setup
    nlon, nlat = 2880, 1441
    lon_edges = np.linspace(-np.pi, np.pi, nlon + 1)
    lon_centers = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    dlon = lon_edges[1] - lon_edges[0]

    lat_centers = np.linspace(-np.pi/2, np.pi/2, nlat)
    dlat = lat_centers[1] - lat_centers[0]
    lat_edges = np.zeros(len(lat_centers) + 1)
    lat_edges[0] = lat_centers[0] - dlat
    lat_edges[1:] = lat_centers

    roi_lon_deg = np.array([roi_center_lon - roi_side_deg / 2, roi_center_lon + roi_side_deg / 2])
    roi_lat_deg = np.array([roi_center_lat - roi_side_deg / 2, roi_center_lat + roi_side_deg / 2])

    # ROI in radians
    roi_lon_rad = np.deg2rad(roi_lon_deg)
    roi_lat_rad = np.deg2rad(roi_lat_deg)

    # Get index ranges for the ROI
    lon_mask = (lon_centers >= roi_lon_rad[0]) & (lon_centers <= roi_lon_rad[1])
    lat_mask = (lat_centers >= roi_lat_rad[0]) & (lat_centers <= roi_lat_rad[1])
    lon_indices = np.where(lon_mask)[0]
    lat_indices = np.where(lat_mask)[0]

    lon_grid_roi, lat_grid_roi = np.meshgrid(
        lon_centers[lon_indices], lat_centers[lat_indices], indexing='ij'
    )

    grid_coords = SkyCoord(lon_grid_roi.ravel() * u.rad,
                           lat_grid_roi.ravel() * u.rad,
                           frame='galactic')

    G_icrs = grid_coords.transform_to('icrs').cartesian.xyz.value.T  # shape (M, 3)
    print(f"G_icrs (ROI) created. Mem: {psutil.Process().memory_info().rss / 1e9:.2f} GB")

    # Satellite state
    pos = np.vstack([df['x'].values, df['y'].values, df['z'].values]).T
    vel = np.vstack([df['vx'].values, df['vy'].values, df['vz'].values]).T
    times = df['utc'].values
    N = len(times)

    suz = -pos / np.linalg.norm(pos, axis=1, keepdims=True)
    vel_proj = np.sum(vel * suz, axis=1, keepdims=True) * suz
    sux = vel - vel_proj
    sux_norm = np.linalg.norm(sux, axis=1, keepdims=True)
    too_small = sux_norm[:, 0] < 1e-5
    fallback_axis = np.cross(np.array([[1, 0, 0]]), suz[too_small])
    sux[too_small] = fallback_axis
    sux /= np.linalg.norm(sux, axis=1, keepdims=True)
    suy = np.cross(suz, sux)
    sux = np.cross(suy, suz)  # Re-orthogonalize

    exposure_map_flat = np.zeros(G_icrs.shape[0], dtype=np.float64)
    batch_size = 1000
    for start_idx in range(0, N, batch_size):
        end_idx = min(start_idx + batch_size, N)
        B = -suz[start_idx:end_idx]
        BX = sux[start_idx:end_idx]
        BY = suy[start_idx:end_idx]

        cos_theta = B @ G_icrs.T
        cos_theta = cos_theta.astype(np.float32)
        mask = cos_theta >= np.cos(np.radians(fov_angle_deg))
        if not np.any(mask):
            continue

        t_dot_sux = (BX @ G_icrs.T).astype(np.float32)
        t_dot_suy = (BY @ G_icrs.T).astype(np.float32)
        phi = np.rad2deg(np.mod(np.arctan2(t_dot_suy, t_dot_sux), 2 * np.pi))

        increments = np.zeros_like(cos_theta, dtype=np.float32)
        masked_theta = cos_theta[mask]
        masked_phi = phi[mask]
        increments[mask] = effective_area(masked_theta, masked_phi, energy)
        exposure_map_flat += increments.sum(axis=0)
        print(f"Processed {start_idx}–{end_idx - 1}. Mem: {psutil.Process().memory_info().rss / 1e9:.2f} GB")

    exposure_map = exposure_map_flat.reshape(lon_grid_roi.shape)
    return {
        "exposure": exposure_map,
        "lon_indices" : lon_indices,
        "lat_indices" : lat_indices,
        "lon_grid": lon_grid_roi,
        "lat_grid": lat_grid_roi,
        "lon_centers": lon_centers[lon_indices],
        "lat_centers": lat_centers[lat_indices],
        "lon_edges": lon_edges[lon_indices[0]:lon_indices[-1] + 2],
        "lat_edges": lat_edges[lat_indices[0]:lat_indices[-1] + 2],
    }

def plot_exposure_map(lon_centers, lat_centers, exposure_map, target_lon, target_lat, target_name="Target", figPath=None):
    """ Plot the cropped exposure map in rectangular projection.
    Args:
        lon_centers (np.ndarray): 1D array of galactic longitudes in radians.
        lat_centers (np.ndarray): 1D array of galactic latitudes in radians.
        exposure_map (np.ndarray): 2D array of exposure values.
        figPath (str, optional): Path to save the figure. If None, the figure is shown.
    """

    # Convert radians to degrees for plotting
    lon_deg = np.rad2deg(lon_centers)
    lat_deg = np.rad2deg(lat_centers)

    # 2D grid for plotting
    lon_grid, lat_grid = np.meshgrid(lon_deg, lat_deg, indexing='ij')

    # Transpose for imshow-like orientation
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.pcolormesh(lon_grid, lat_grid, exposure_map, cmap='viridis', shading='auto')
    ax.plot(target_lon, target_lat, 'rx', markersize=8, label=f'{target_name}')
    ax.legend(loc='upper right')

    ax.set_xlabel('Galactic Longitude [deg]')
    ax.set_ylabel('Galactic Latitude [deg]')
    ax.set_title('Cropped Exposure Map')

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Exposure [cm² s sr]')

    ax.set_aspect('auto')
    ax.grid(True)

    if figPath:
        plt.savefig(figPath)
        plt.close(fig)
    else:
        plt.show()

In [20]:
date_str = "2021-01-01"
energy_bin = 7
target = "Vela"

energy = ENERGY_BINS_CENTERS[int(energy_bin)]  # Convert energy bin to GeV
day_start = Time(f"{date_str}T00:00:00", scale='utc')
day_end = day_start + TimeDelta(1, format='jd')

# File pattern for current and next day
year, month, day = date_str.split("-")
base = "/dpnc/beegfs/dampe/prod/FM/skim/6.0.0/v2"
current_pattern = f"{base}/{year}/{month}/{day}_data_photon.root"
next_day_time = (day_start + TimeDelta(1, format='jd')).to_datetime(leap_second_strict='warn')
next_pattern = f"{base}/{next_day_time.year:04d}/{next_day_time.month:02d}/{next_day_time.day:02d}_data_photon.root"

figPath_base = f"/home/users/l/lebrun/figures/{target}/exposureMaps/{energy:.3f}GeV/{year}/{month}/"

try :
    if not os.path.exists(figPath_base):
        os.makedirs(figPath_base)
except FileExistsError as e:
    pass

figPath = figPath_base + f"{date_str}.png"

resultPath_base = f"/dpnc/beegfs/dampe/tryf/ForKimberly/exposureMaps_GIE/{energy:.3f}GeV/daily/"

try : 
    if not os.path.exists(resultPath_base):
        os.makedirs(resultPath_base)
except FileExistsError as e:
    pass

resultPath = resultPath_base + f"{date_str}.npz"

# Load target coordinates
# Main extraction code
dirDir= "/home/users/l/lebrun/"
parDirName =  dirDir + f"data/{target}/"

# HERE you can enter your custom path to a .par file. Otherwise the script will look
# into data/{target} for a .par file
filename = None 

if filename is None:
    try:
        for file in os.listdir(parDirName[:-1]):
            if file.endswith(".par"):
                filename = os.path.join(parDirName, file)
                break
    except FileNotFoundError as e:
        print(f"Error: no par file found in subdirectory {parDirName}. Target: {target}")
        raise e

if filename is None:
    raise FileNotFoundError(f"No .par file found in {parDirName}")

# Extract RAJ and DECJ from the par file
raj_str, decj_str = parse_par_file(filename)

if raj_str is None or decj_str is None:
    print("Error: Could not find RAJ and/or DECJ in the .par file")
    print(f"RAJ: {raj_str}, DECJ: {decj_str}")
else:
    # Convert to decimal values
    ra_target = convert_ra_to_decimal(raj_str)
    dec_target = convert_dec_to_decimal(decj_str)
    
    print(f"RAJ: {ra_target} hours, DECJ: {dec_target} degrees")
    print(f"Original format - RAJ: {raj_str}, DECJ: {decj_str}")
    
    # Create SkyCoord object (same as original)
    if ra_target is not None and dec_target is not None:
        c = SkyCoord(ra=ra_target*u.hourangle, dec=dec_target*u.degree, frame='icrs')
        c = c.transform_to('galactic')
        print(f"Galactic coordinates: l={c.l}, b={c.b}")

target_lon = c.l.wrap_at(180 * u.deg).degree  # Wrap longitude to [-180, 180]
target_lat = c.b.degree  # Latitude in degrees

print(f"Target coordinates: lon={target_lon}, lat={target_lat}")


RAJ: 8.589058747222223 hours, DECJ: -45.17635419444444 degrees
Original format - RAJ: 8:35:20.61149000, DECJ: -45:10:34.87510000
Galactic coordinates: l=263.5518314289383 deg, b=-2.7873123683789167 deg
Target coordinates: lon=-96.44816857106167, lat=-2.7873123683789167


In [21]:
df = load_filtered_day_dataframe(current_pattern, next_pattern, day_start, day_end)
print(f"Loaded {len(df)} timestamps for {date_str}.")

# Print start and end timestamps
start_time = met_to_utc(df['second'].iloc[0], df['microsecond'].iloc[0])
end_time = met_to_utc(df['second'].iloc[-1], df['microsecond'].iloc[-1])
print(f"Start time: {start_time.iso}, End time: {end_time.iso}")

df, mask = mask_saa(df)

print(f"Masked out {np.sum(~mask)} timestamps inside SAA.")
res = compute_exposure_map_roi(df, energy, target_lon, target_lat)
lon_edges = res['lon_edges']
lat_edges = res['lat_edges']
lon_centers = res['lon_centers']
lat_centers = res['lat_centers']
exposure_map = res['exposure']

fig = plot_exposure_map(lon_centers, lat_centers, exposure_map, target_lon, target_lat, target_name=target, figPath=figPath)
print(f"Exposure map computed for {date_str} and saved to {figPath}.")

Loaded 86128 timestamps for 2021-01-01.
Start time: 2021-01-01 00:03:54.750, End time: 2021-01-01 23:59:59.750


/tmp/ipykernel_2468026/2869832833.py:42: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  mask_inside_saa = vectorized.contains(saa_polygon_shape, lon, lat)


Masked out 4181 timestamps inside SAA.
G_icrs (ROI) created. Mem: 1.32 GB
Processed 2000–2999. Mem: 1.38 GB
Processed 3000–3999. Mem: 1.42 GB
Processed 4000–4999. Mem: 1.42 GB
Processed 8000–8999. Mem: 1.42 GB
Processed 9000–9999. Mem: 1.42 GB
Processed 13000–13999. Mem: 1.42 GB
Processed 14000–14999. Mem: 1.42 GB
Processed 15000–15999. Mem: 1.42 GB
Processed 19000–19999. Mem: 1.42 GB
Processed 20000–20999. Mem: 1.42 GB
Processed 21000–21999. Mem: 1.42 GB
Processed 24000–24999. Mem: 1.42 GB
Processed 25000–25999. Mem: 1.42 GB
Processed 26000–26999. Mem: 1.42 GB
Processed 29000–29999. Mem: 1.42 GB
Processed 30000–30999. Mem: 1.42 GB
Processed 34000–34999. Mem: 1.42 GB
Processed 35000–35999. Mem: 1.41 GB
Processed 36000–36999. Mem: 1.41 GB
Processed 39000–39999. Mem: 1.41 GB
Processed 40000–40999. Mem: 1.41 GB
Processed 41000–41999. Mem: 1.41 GB
Processed 45000–45999. Mem: 1.41 GB
Processed 46000–46999. Mem: 1.41 GB
Processed 47000–47999. Mem: 1.41 GB
Processed 51000–51999. Mem: 1.41 GB
